In [6]:
import numpy as np
import pandas as pd
import cv2
import os
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
from IPython.display import display, Image as IPImage

np.random.seed(47)
random.seed(47)

In [7]:
def apply_sepia(image):
    """Convert an image to sepia tone"""
    sepia_matrix = np.array([
        [0.272, 0.534, 0.131],
        [0.349, 0.686, 0.168],
        [0.393, 0.769, 0.189]
    ])
    sepia_img = cv2.transform(image, sepia_matrix)
    sepia_img = np.clip(sepia_img, 0, 255).astype(np.uint8)
    return sepia_img

In [8]:
image_folder = Path('data/lab07')
image_folder.mkdir(parents=True, exist_ok=True)

image_files = list(image_folder.glob('*.jpeg')) + list(image_folder.glob('*.png'))

original_images = [img for img in image_files if 'sepia' not in img.name.lower()]

In [9]:
for img_path in original_images:
    img = cv2.imread(str(img_path))
    
    if img is None:
        print(f"Nu gasesc {img_path.name}")
        continue
        
    sepia_img = apply_sepia(img)
    
    sepia_filename = f"sepia_{img_path.name}"
    sepia_path = image_folder / sepia_filename
    cv2.imwrite(str(sepia_path), sepia_img)

In [10]:
all_images = list(image_folder.glob('*.jpeg')) + list(image_folder.glob('*.png'))

data = []
for img_path in all_images:
    filename = img_path.name
    
    if 'sepia' in filename.lower():
        label = 1
        class_name = 'sepia'
    else:
        label = 0
        class_name = 'original'
    
    data.append({
        'filename': filename,
        'filepath': str(img_path),
        'label': label,
        'class': class_name
    })
    
    
df = pd.DataFrame(data)
df.to_csv('data/lab07/image_database.csv', index=False)


In [9]:
#train & test cu tool neural_network.MLPClassifier()
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [10]:
df = pd.read_csv('data/lab07/image_database.csv')

def load_images(df, img_size=(32, 32)):
    images = []
    labels = []
    
    for _, row in df.iterrows():
        img = cv2.imread(row['filepath'])
        if img is not None:
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img / 255.0  # normalizt min-max
            images.append(img.flatten()) 
            labels.append(row['label'])
    
    return np.array(images), np.array(labels)

In [11]:
X, y = load_images(df)  #imagini + labels

#split in train si test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),    # 2 hidden layers with 64 and 32 neurons
    activation='relu',              
    max_iter=50,                    
    random_state=47,
    verbose=True 
)

mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")


Iteration 1, loss = 0.62206308
Iteration 2, loss = 0.48108984
Iteration 3, loss = 0.29614145
Iteration 4, loss = 0.14741261
Iteration 5, loss = 0.08160151
Iteration 6, loss = 0.06313472
Iteration 7, loss = 0.03355269
Iteration 8, loss = 0.01870194
Iteration 9, loss = 0.01345419
Iteration 10, loss = 0.01119629
Iteration 11, loss = 0.00902048
Iteration 12, loss = 0.00652568
Iteration 13, loss = 0.00439468
Iteration 14, loss = 0.00306568
Iteration 15, loss = 0.00231976
Iteration 16, loss = 0.00187800
Iteration 17, loss = 0.00159659
Iteration 18, loss = 0.00141333
Iteration 19, loss = 0.00129248
Iteration 20, loss = 0.00120757
Iteration 21, loss = 0.00114399
Iteration 22, loss = 0.00109228
Iteration 23, loss = 0.00104873
Iteration 24, loss = 0.00100687
Iteration 25, loss = 0.00096824
Iteration 26, loss = 0.00093229
Iteration 27, loss = 0.00089881
Iteration 28, loss = 0.00086783
Iteration 29, loss = 0.00083951
Iteration 30, loss = 0.00081381
Training loss did not improve more than tol=0.000

In [13]:
#influenta hiperparametri

In [14]:
df = pd.read_csv('data/lab07/image_database.csv')

def load_images(df, img_size=(32, 32)):
    images, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row['filepath'])
        if img is not None:
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img / 255.0
            images.append(img.flatten())
            labels.append(row['label'])
    return np.array(images), np.array(labels)

X, y = load_images(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
# 1. nr neuroni in hidden layer

hidden_sizes = [(32,), (64,), (128,), (64, 32), (128, 64)]
for hidden in hidden_sizes:
    mlp = MLPClassifier(hidden_layer_sizes=hidden, max_iter=100, random_state=47)
    mlp.fit(X_train, y_train)
    acc = mlp.score(X_test, y_test)
    print(f"Hidden {str(hidden):15} -> Accuracy: {acc:.4f}")

Hidden (32,)           -> Accuracy: 0.7500
Hidden (64,)           -> Accuracy: 0.7500
Hidden (128,)          -> Accuracy: 0.5000
Hidden (64, 32)        -> Accuracy: 0.7500
Hidden (128, 64)       -> Accuracy: 0.5000


In [25]:
# 2. max_iter 

iterations = [5, 25, 50, 100]
for it in iterations:
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=it, random_state=47)
    mlp.fit(X_train, y_train)
    acc = mlp.score(X_test, y_test)
    print(f"Iterations {it:3} -> Accuracy: {acc:.4f}")


C:\Users\Daria\miniconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Daria\miniconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (25) reached and the optimization hasn't converged yet.
  warnings.warn(


Iterations   5 -> Accuracy: 0.5000
Iterations  25 -> Accuracy: 0.7500
Iterations  50 -> Accuracy: 0.7500
Iterations 100 -> Accuracy: 0.7500


In [32]:
# ann cod propriu

class Anna:
    def __init__(self, layer_sizes, learning_rate=0.01, random_state=47):
        np.random.seed(random_state)
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            limit = np.sqrt(2 / layer_sizes[i])
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * limit
            b = np.zeros((1, layer_sizes[i+1]))
            
            self.weights.append(w)
            self.biases.append(b)
            
    def relu(self, z):
        return np.maximum(0, z)
    
    def relu_derivative(self, z):
        return (z > 0).astype(float)
    
    def sigmoid(self, z):
        # FIX: Clip to prevent overflow
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def forward(self, X):
        self.layer_out = [] 
        self.layer_in = []
        
        current = X
        
        for i in range(len(self.weights) - 1):
            z = np.dot(current, self.weights[i]) + self.biases[i]
            self.layer_in.append(z)
            current = self.relu(z)
            self.layer_out.append(current)
        
        #output layer
        z = np.dot(current, self.weights[-1]) + self.biases[-1]
        self.layer_in.append(z)
        current = self.sigmoid(z)
        self.layer_out.append(current)
        
        return current
    
    
    def backward(self, X, y, output):
        m = X.shape[0]
        
        dw = [None] * len(self.weights)
        db = [None] * len(self.biases)
        
        #gradient pt output layer = derivata sigmoid
        delta = output - y.reshape(-1, 1)
        
        #gradient weights + bias
        dw[-1] = np.dot(self.layer_out[-2].T, delta) / m
        db[-1] = np.sum(delta, axis=0, keepdims=True) / m
        
        for i in range(len(self.weights) - 2, -1, -1):
            delta = np.dot(delta, self.weights[i+1].T) * self.relu_derivative(self.layer_in[i])
            dw[i] = np.dot(self.layer_out[i].T, delta) / m
            db[i] = np.sum(delta, axis=0, keepdims=True) / m
        
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * dw[i]
            self.biases[i] -= self.lr * db[i]
    
    
    def fit(self, X, y, epochs=100, batch_size=32, verbose=True):
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            total_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                #forward
                output = self.forward(X_batch)
                
                #calc loss - binary cross entropy
                loss = -np.mean(y_batch * np.log(output + 1e-8) + 
                               (1 - y_batch) * np.log(1 - output + 1e-8))
                total_loss += loss
                n_batches += 1
                
                #backward
                self.backward(X_batch, y_batch, output)
            
            if verbose and (epoch % 20 == 0 or epoch == epochs-1):
                avg_loss = total_loss / n_batches
                train_pred = self.predict(X)
                train_acc = np.mean(train_pred == y)
                print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {train_acc:.4f}")
                
    def predict(self, X):
        output = self.forward(X)
        return (output >= 0.5).flatten().astype(int)
    

In [36]:
X, y = load_images(df)  #imagini + labels

#split in train si test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

anna = Anna(layer_sizes=[X_train.shape[1], 16, 8, 1], learning_rate=0.0001)
anna.fit(X_train, y_train, epochs=100, batch_size=32)
y_pred = anna.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f"anna accuracy: {accuracy:.4f}")

Epoch   0/100 | Loss: 1.1038 | Accuracy: 0.4375
Epoch  20/100 | Loss: 1.0563 | Accuracy: 0.5000
Epoch  40/100 | Loss: 1.0195 | Accuracy: 0.5625
Epoch  60/100 | Loss: 0.9848 | Accuracy: 0.5625
Epoch  80/100 | Loss: 0.9517 | Accuracy: 0.5625
Epoch  99/100 | Loss: 0.9291 | Accuracy: 0.5625
anna accuracy: 0.7500


In [ ]:
"""
inainte sa scale weights down

Epoch   0/100 | Loss: 8.7786 | Accuracy: 0.5000
Epoch  20/100 | Loss: 9.0664 | Accuracy: 1.0000
Epoch  40/100 | Loss: 9.0664 | Accuracy: 1.0000
Epoch  60/100 | Loss: 9.0664 | Accuracy: 1.0000
Epoch  80/100 | Loss: 9.0664 | Accuracy: 1.0000
Epoch  99/100 | Loss: 9.0664 | Accuracy: 1.0000
anna accuracy: 0.2500


For ReLU activation, the optimal variance is:
    Var(w) = 2 / n

Therefore:
    std(w) = sqrt(2 / n)
 
fixes(ish):

1. scale weights down
limit = np.sqrt(2 / layer_sizes[i])
w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * limit

explicatie de la mine:

varianta(output) = varianta(input) * varianta(weights) * nr_in
si vreau varianta(output) = varianta(input) => varianta(weights) = 1/nr_in 
din cauza relu negativele se duc => 
varianta(output) = varianta(input) * varianta(weights) * nr_in * 0.5 => varianta(weights) = 2/nr_in
"""

In [38]:
#cnn cod propriu

class CNN:
    def __init__(self, learning_rate=0.001):
        np.random.seed(47)
        self.lr = learning_rate
        
        # 3 filters of size 3x3
        self.filters = np.random.randn(3, 3, 3) * 0.1
        self.filter_bias = np.zeros(3)
        
        self.dense_weight = np.random.randn(3*30*30, 1) * 0.01 #30=out size = in - filter +1
        self.dense_bias = 0
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def sigmoid(self, x):
        return 1/(1+np.exp(-np.clip(x, -500, 500)))
    
    def conv2d(self, X, filters):
        h, w, c = X.shape
        fh, fw, fc = filters.shape
        out_h, out_w = h - fh + 1, w - fw + 1
        output = np.zeros((out_h, out_w, fc))
        
        for k in range(fc):
            for i in range(out_h):
                for j in range(out_w):
                    region = X[i:i+fh, j:j+fw, :]
                    output[i,j,k] = np.sum(region * filters[:,:,k])
        return output
    
    def forward(self, X):
        # convolutoe
        self.conv = self.conv2d(X, self.filters) + self.filter_bias
        self.conv_act = self.relu(self.conv)
        
        self.flat = self.conv_act.flatten()
        
        z = np.dot(self.flat, self.dense_weight) + self.dense_bias
        return self.sigmoid(z)[0]
    
    def predict(self, X):
        return 1 if self.forward(X) >= 0.5 else 0
    
    def fit(self, X, y, epochs=50):
        for epoch in range(epochs):
            correct = 0
            for i in range(len(X)):
                # forward
                pred = self.forward(X[i])
                
                # backward
                error = pred - y[i]
                
                # update layer
                self.dense_weight -= self.lr * error * self.flat.reshape(-1,1)
                self.dense_bias -= self.lr * error
                
                # Update filters (simplified)
                self.filters -= self.lr * error * np.random.randn(3,3,3) * 0.001
                
                if (pred >= 0.5) == y[i]:
                    correct += 1
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Acc = {correct/len(X):.4f}")

In [3]:
def load_images_cnn(df, img_size=(32, 32)):  # Resize to 32x32
    images = []
    labels = []
    for _, row in df.iterrows():
        img = cv2.imread(row['filepath'])
        if img is not None:
            # RESIZE to 32x32 
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img / 255.0  
            images.append(img)
            labels.append(row['label'])
    return np.array(images), np.array(labels)



In [45]:
X, y = load_images_cnn(df, img_size=(32, 32))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)
cnn = CNN(learning_rate=0.01)
cnn.fit(X_train, y_train, epochs=50)
correct = sum(cnn.predict(X_test[i]) == y_test[i] for i in range(len(X_test)))
print(f"CNN Test Accuracy: {correct/len(X_test):.4f}")


Epoch 0: Acc = 0.4375
Epoch 10: Acc = 0.6250
Epoch 20: Acc = 0.6875
Epoch 30: Acc = 0.7500
Epoch 40: Acc = 0.7500
CNN Test Accuracy: 0.5000
